# Training: DUDES Student

Trains a DUDES student for each **(arch × test year)** using the fair ensemble
(3 models that never saw the test year) as the teacher.

The **backbone fold** (`bb_fold`) is the ensemble member with the **median** test AP
among the 3 fair folds (`BB_FOLD_SELECTION = "middle"`).
The data split uses `data_fold_id=bb_fold` so the frozen backbone sees the
**exact same distribution** it was originally trained on.

| Test Year | Teacher Folds | Backbone fold       | Data split matches backbone |
|-----------|--------------|---------------------|----------------------------|
| 2018 | 7, 9, 11 | median-AP of {7,9,11} | yes |
| 2019 | 3, 5, 10 | median-AP of {3,5,10} | yes |
| 2020 | 1, 4, 8  | median-AP of {1,4,8}  | yes |
| 2021 | 0, 2, 6  | median-AP of {0,2,6}  | yes |

**Cells:**
1. Configuration 
2. Helper Functions
3. Run Training


### 1. Configuration

In [ ]:
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────
BASE_DIR      = Path().resolve().parent          
_WEIGHTS_ROOT = BASE_DIR / "pretrained_weights"
DATA_DIR      = BASE_DIR / "WildfireSpreadTS_HDF5"
OUTPUT_DIR    = BASE_DIR / "results" / "DUDES_Training_UTAE_T5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Data ──────────────────────────────────────────────────────
LOAD_HDF5     = True
WSTS_CHANNELS = [0, 1, 2, 3, 4, 38, 39]   # vegetation features (WildfireSpreadTS indices)

# ── Architecture registry ─────────────────────────────────────
ARCH_CONFIG = {
    "utae_t5": {
        "weights_dir":      _WEIGHTS_ROOT / "Res18UTAE_T5" / "Veg",
        "n_timesteps":      5,
        "model_class":      "SMPTempModel",
        "flatten_temporal": False,
    },
    "unet_t5": {
        "weights_dir":      _WEIGHTS_ROOT / "Res18Unet_T5" / "Veg",
        "n_timesteps":      5,
        "model_class":      "SMPModel",
        "flatten_temporal": True,
    },
    "unet_t1": {
        "weights_dir":      _WEIGHTS_ROOT / "Res18Unet_T1" / "Veg",
        "n_timesteps":      1,
        "model_class":      "SMPModel",
        "flatten_temporal": True,
    },
}

# ARCHS = ["utae_t5", "unet_t5", "unet_t1"]
ARCHS = ["utae_t5"]

# ── Fair ensemble mapping ─────────────────────────────────────
# Keys define the test years; values are the 3 folds that never saw that year.
YEAR_TO_FOLDS = {
    2018: [7, 9, 11],
    2019: [3, 5, 10],
    2020: [1, 4, 8],
    2021: [0, 2, 6],
}
# "best" | "middle" | "worst"  — selects backbone from the 3 fair folds by testAP rank.
# "middle" (median) is robust and avoids selection bias.
BB_FOLD_SELECTION = "middle"

# ── Training hyperparameters ──────────────────────────────────
BATCH_SIZE        = 4
NUM_WORKERS       = 0
MAX_EPOCHS        = 100
PATIENCE          = 20
LR                = 1e-3
ASD_RADIUS_PX     = 4    # circular dilation around GT fire mask used for val_unc_auroc

print(f"Architectures    : {ARCHS}")
print(f"Test years       : {sorted(YEAR_TO_FOLDS)}")
print(f"Ensemble map     : {YEAR_TO_FOLDS}")
print(f"BB fold strategy : {BB_FOLD_SELECTION}")
print(f"ASD radius       : {ASD_RADIUS_PX}px")
print(f"Data dir         : {DATA_DIR}")
print(f"Output dir       : {OUTPUT_DIR}")


### 2. Helper Functions

In [ ]:
import json
import re
import sys
import time
import types

import numpy as np
import torch
import torch.nn as nn
import pytorch_lightning as pl
import torchmetrics
from scipy.ndimage import binary_dilation as _sp_binary_dilation

sys.modules.setdefault("wandb", types.ModuleType("wandb"))

for p in [str(BASE_DIR / "third_party"), str(BASE_DIR / "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

from models.SMPModel     import SMPModel
from models.SMPTempModel import SMPTempModel
from dataloader.FireSpreadDataModule import FireSpreadDataModule
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cpu")
print(f"Device: {device}")


# ── Parse AP from checkpoint filename ─────────────────────────
def _parse_ap_from_path(p):
    m = re.search(r"testAP(\d+\.\d+)", p.name)
    return float(m.group(1)) if m else -1.0


def resolve_bb_fold(pth_paths, year):
    fold_ids = YEAR_TO_FOLDS[year]
    ranked = sorted(fold_ids, key=lambda fid: _parse_ap_from_path(pth_paths[fid]))
    if BB_FOLD_SELECTION == "best":
        fid = ranked[-1]
    elif BB_FOLD_SELECTION == "worst":
        fid = ranked[0]
    else:
        fid = ranked[len(ranked) // 2]
    return fid, _parse_ap_from_path(pth_paths[fid])


def load_model_for_arch(arch: str, pth_path: str):
    cfg   = ARCH_CONFIG[arch]
    state = torch.load(pth_path, map_location=device, weights_only=True)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    n_channels = next(
        v.shape[1] for k, v in state.items() if k.endswith("encoder.conv1.weight")
    )
    if cfg["model_class"] == "SMPTempModel":
        model = SMPTempModel(
            encoder_name="resnet18", n_channels=n_channels,
            flatten_temporal_dimension=cfg["flatten_temporal"],
            pos_class_weight=10.0, encoder_weights=None, loss_function="BCE",
        )
    else:
        model = SMPModel(
            encoder_name="resnet18", n_channels=n_channels,
            flatten_temporal_dimension=cfg["flatten_temporal"],
            pos_class_weight=10.0, encoder_weights=None, loss_function="BCE",
        )
    nn.Module.load_state_dict(model, state, strict=False)
    return model.eval().to(device)


def det_forward(model, x: torch.Tensor) -> torch.Tensor:
    """Return raw logits (no sigmoid) for a single model."""
    model.eval()
    with torch.no_grad():
        return model(x).squeeze(1)


# Maximum possible sample std for N=3 sigmoid outputs in [0, 1].
# Achieved at configurations {0, 0, 1} or {0, 1, 1}:
#   s = sqrt( k*(N-k) / (N*(N-1)) )  with k=1, N=3  →  sqrt(1/3) = 1/sqrt(3) ≈ 0.5774
# Dividing by this maps the teacher's full std range into [0, 1] without premature clamping.
_TEACHER_STD_MAX = (1.0 / 3.0) ** 0.5  # = 1/sqrt(3) ≈ 0.5774

def ensemble_uncertainty(logit_list):
    """Epistemic uncertainty: std of sigmoid predictions, normalised to [0, 1].

    Dividing by _TEACHER_STD_MAX ensures the teacher targets span the full [0, 1]
    range that the student's Sigmoid head can express — matching DUDES design intent.
    """
    probs = torch.stack([torch.sigmoid(l) for l in logit_list], dim=0)
    return (probs.std(dim=0) / _TEACHER_STD_MAX).clamp(0.0, 1.0)


# ── PolyLR ────────────────────────────────────────────────────
class PolyLR(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, max_iterations, power=0.9, last_epoch=-1):
        self.max_iterations = max_iterations
        self.power          = power
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        factor = (1 - self.last_epoch / self.max_iterations) ** self.power
        return [base_lr * max(factor, 0.0) for base_lr in self.base_lrs]


# ── EpochLogger ───────────────────────────────────────────────
class EpochLogger(pl.Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        nan = float("nan")
        print(
            f"  Epoch {trainer.current_epoch:3d}  "
            f"| train  loss={m.get('train_loss_epoch', nan):.4f}"
            f"  | val  loss={m.get('val_loss', nan):.4f}"
            f"  | auroc={m.get('val_unc_auroc', nan):.4f}"
        )


# ── ModelWithUncertaintyHead ───────────────────────────────────
class ModelWithUncertaintyHead(nn.Module):
    """Frozen backbone + uncertainty head. Used only for caching.

    Head: 1x1 conv + Sigmoid — matches the original DUDES SegmentationHead.
    Per-pixel prediction with no added spatial mixing (the decoder already
    provides spatial context), minimising overfitting on the small wildfire dataset.
    """
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        dec_out_ch    = backbone.model.segmentation_head[0].in_channels
        self.uncertainty_head = nn.Sequential(
            nn.Conv2d(dec_out_ch, 1, kernel_size=1),
            nn.Sigmoid(),
        )
        self._decoder_out = None
        backbone.model.decoder.register_forward_hook(self._hook)

    def _hook(self, module, input, output):
        self._decoder_out = output

    def forward(self, x):
        seg_logit   = self.backbone(x)
        uncertainty = self.uncertainty_head(self._decoder_out)
        return seg_logit, uncertainty


# ── Losses ────────────────────────────────────────────────────
class RMSLELoss(nn.Module):
    def forward(self, pred, target):
        return torch.sqrt(torch.mean((torch.log1p(pred) - torch.log1p(target)) ** 2))


# ── CachedDUDESStudent ────────────────────────────────────────
class CachedDUDESStudent(pl.LightningModule):
    """Trains the uncertainty head on cached decoder features.

    Head:      1x1 conv + Sigmoid (matches original DUDES SegmentationHead)
    Loss:      RMSLE
    Monitored: val_unc_auroc — AUROC of uncertainty predicting errors,
               restricted to a circular dilation zone around the GT fire mask
               (radius = ASD_RADIUS_PX from Cell 1 config).
    """
    def __init__(
        self,
        dec_out_ch: int,
        lr:         float,
        max_steps:  int,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.uncertainty_head = nn.Sequential(
            nn.Conv2d(dec_out_ch, 1, kernel_size=1),
            nn.Sigmoid(),
        )
        self._rmsle          = RMSLELoss()
        self._val_unc_auroc  = torchmetrics.AUROC(task="binary")
        self._auroc_has_data = False
        # Circular structuring element — radius from Cell 1 config
        _r = np.arange(-ASD_RADIUS_PX, ASD_RADIUS_PX + 1)
        self._disk = (_r[:, None]**2 + _r[None, :]**2) <= ASD_RADIUS_PX**2

    def on_validation_epoch_start(self):
        self._val_unc_auroc.reset()
        self._auroc_has_data = False

    def training_step(self, batch, batch_idx):
        dec_feat, seg_logit, y, teacher_unc = batch
        unc  = self.uncertainty_head(dec_feat).squeeze(1)
        loss = self._rmsle(unc, teacher_unc)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        dec_feat, seg_logit, y, teacher_unc = batch
        unc       = self.uncertainty_head(dec_feat).squeeze(1)
        seg_logit = seg_logit.squeeze(1)
        y_f       = y.float()
        loss      = self._rmsle(unc, teacher_unc)

        with torch.no_grad():
            pred_binary = (torch.sigmoid(seg_logit) > 0.5).long()
            error_map   = (pred_binary != y_f.long())

            y_np     = y_f.cpu().numpy().astype(bool)
            buf_list = [
                _sp_binary_dilation(y_np[b], structure=self._disk)
                for b in range(y_np.shape[0])
            ]
            buf_mask = torch.from_numpy(np.stack(buf_list))

            unc_buf = unc[buf_mask].flatten()
            err_buf = error_map[buf_mask].long().flatten()

            if err_buf.sum() > 0 and (1 - err_buf).sum() > 0:
                self._val_unc_auroc.update(unc_buf, err_buf)
                self._auroc_has_data = True

        self.log("val_loss", loss, on_epoch=True, prog_bar=True)
        return loss

    def on_validation_epoch_end(self):
        if self._auroc_has_data:
            auroc = self._val_unc_auroc.compute()
        else:
            auroc = torch.tensor(0.5)
        self.log("val_unc_auroc", auroc, prog_bar=True)

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(
            self.parameters(), lr=self.hparams.lr,
            momentum=0.9, weight_decay=5e-4,
        )
        scheduler = PolyLR(optimizer, max_iterations=self.hparams.max_steps, power=0.9)
        return [optimizer], [{"scheduler": scheduler, "interval": "step"}]


print("Helpers ready.")
print(f"Teacher std normalisation constant : {_TEACHER_STD_MAX:.4f}")
print(f"Validation ASD radius              : {ASD_RADIUS_PX}px")


### 3. Run Training

In [ ]:
results_summary = []

for ARCH in ARCHS:
    _cfg   = ARCH_CONFIG[ARCH]
    _n_obs = _cfg["n_timesteps"]

    pth_paths = sorted(
        _cfg["weights_dir"].glob("fold*.pth"),
        key=lambda p: int(p.stem.split("_")[0].replace("fold", "")),
    )

    print(f"\n{'#'*64}")
    print(f"  Student training — {ARCH}  (cached features)")
    print(f"{'#'*64}")

    for YEAR in sorted(YEAR_TO_FOLDS):
        bb_fold, bb_ap = resolve_bb_fold(pth_paths, YEAR)
        fold_ids       = YEAR_TO_FOLDS[YEAR]
        ckpt_dir       = OUTPUT_DIR / f"{ARCH}_year{YEAR}"

        # ── Skip if head checkpoint already exists ────────────
        existing_ckpts = sorted(ckpt_dir.glob("head_*.pt")) if ckpt_dir.exists() else []
        if existing_ckpts:
            def _parse_ckpt_skip(p):
                m = re.search(r"vauroc=(\d+\.\d+)", p.name)
                return float(m.group(1)) if m else -1.0
            best_existing        = max(existing_ckpts, key=_parse_ckpt_skip)
            best_existing_vauroc = _parse_ckpt_skip(best_existing)
            print(f"\n  {ARCH} year {YEAR}: checkpoint exists "
                  f"(val_unc_auroc={best_existing_vauroc:.4f}), skipping")
            results_summary.append({
                "arch": ARCH, "test_year": YEAR,
                "backbone_fold": bb_fold, "backbone_ap": bb_ap,
                "best_val_auroc": best_existing_vauroc,
                "checkpoint": str(best_existing),
            })
            continue

        print(f"\n{'='*56}")
        print(f"  {ARCH}  test year {YEAR}  (backbone fold {bb_fold}, testAP={bb_ap:.3f})")
        print(f"{'='*56}")

        backbone_pth = str(pth_paths[bb_fold])
        print(f"  Backbone: {Path(backbone_pth).name}")

        dm = FireSpreadDataModule(
            data_dir=str(DATA_DIR),
            batch_size=BATCH_SIZE,
            n_leading_observations=_n_obs,
            crop_side_length=128,
            load_from_hdf5=LOAD_HDF5,
            num_workers=NUM_WORKERS,
            remove_duplicate_features=False,
            features_to_keep=WSTS_CHANNELS,
            n_leading_observations_test_adjustment=_n_obs,
            data_fold_id=bb_fold,
            return_doy=False,
        )
        dm.setup("fit")

        # ── Load teachers + backbone ──────────────────────────
        t0 = time.time()
        print(f"  Loading 3 teachers (folds {fold_ids}) + backbone (fold {bb_fold})...")
        teachers = [load_model_for_arch(ARCH, str(pth_paths[f])) for f in fold_ids]
        backbone = load_model_for_arch(ARCH, backbone_pth)
        wrapped  = ModelWithUncertaintyHead(backbone)
        wrapped.eval()

        # ── Cache train features + teacher uncertainty ────────
        print("  Caching train features...")
        _loader = DataLoader(dm.train_dataset, batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=0, pin_memory=False)
        c_dec, c_seg, c_y, c_unc = [], [], [], []
        with torch.no_grad():
            for i, batch in enumerate(_loader):
                x      = batch[0].to(device)
                logits = [det_forward(m, x) for m in teachers]
                unc    = ensemble_uncertainty(logits)
                seg_logit, _ = wrapped(x)
                c_dec.append(wrapped._decoder_out.cpu())
                c_seg.append(seg_logit.cpu())
                c_y.append(batch[1].float().squeeze(1).cpu())
                c_unc.append(unc.cpu())
                if (i + 1) % 50 == 0:
                    print(f"    train {i+1}/{len(_loader)}")

        train_dec     = torch.cat(c_dec)
        train_seg     = torch.cat(c_seg)
        train_y       = torch.cat(c_y)
        teacher_unc_t = torch.cat(c_unc)
        del c_dec, c_seg, c_y, c_unc
        print(f"  Train: {len(train_dec)} samples, mean_unc={teacher_unc_t.mean():.4f}")

        # ── Cache val features + teacher uncertainty ──────────
        print("  Caching val features...")
        _vloader = DataLoader(dm.val_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0, pin_memory=False)
        v_dec, v_seg, v_y, v_unc = [], [], [], []
        with torch.no_grad():
            for i, batch in enumerate(_vloader):
                x      = batch[0].to(device)
                logits = [det_forward(m, x) for m in teachers]
                unc    = ensemble_uncertainty(logits)
                seg_logit, _ = wrapped(x)
                v_dec.append(wrapped._decoder_out.cpu())
                v_seg.append(seg_logit.cpu())
                v_y.append(batch[1].float().squeeze(1).cpu())
                v_unc.append(unc.cpu())
                if (i + 1) % 50 == 0:
                    print(f"    val {i+1}/{len(_vloader)}")

        val_dec           = torch.cat(v_dec)
        val_seg           = torch.cat(v_seg)
        val_y             = torch.cat(v_y)
        val_teacher_unc_t = torch.cat(v_unc)
        del v_dec, v_seg, v_y, v_unc
        print(f"  Val: {len(val_dec)} samples, mean_unc={val_teacher_unc_t.mean():.4f}")

        dec_out_ch = int(train_dec.shape[1])
        del teachers, backbone, wrapped, dm
        print(f"  Cached in {time.time()-t0:.0f}s  (dec_ch={dec_out_ch})")

        # ── Build datasets + loaders ──────────────────────────
        train_ds     = TensorDataset(train_dec, train_seg, train_y, teacher_unc_t)
        val_ds       = TensorDataset(val_dec,   val_seg,   val_y,   val_teacher_unc_t)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
        total_steps  = len(train_loader) * MAX_EPOCHS

        cached_student = CachedDUDESStudent(
            dec_out_ch = dec_out_ch,
            lr         = LR,
            max_steps  = total_steps,
        )

        ckpt_dir.mkdir(parents=True, exist_ok=True)

        ckpt_cb = pl.callbacks.ModelCheckpoint(
            dirpath    = str(ckpt_dir),
            filename   = f"cached_{ARCH}_year{YEAR}" + "-{epoch}-{val_unc_auroc:.4f}",
            monitor    = "val_unc_auroc",
            mode       = "max",
            save_top_k = 1,
        )
        callbacks = [
            EpochLogger(),
            ckpt_cb,
            pl.callbacks.EarlyStopping(
                monitor  = "val_unc_auroc",
                mode     = "max",
                patience = PATIENCE,
            ),
        ]

        trainer = pl.Trainer(
            accelerator         = "cpu",
            devices             = 1,
            max_epochs          = MAX_EPOCHS,
            callbacks           = callbacks,
            logger              = False,
            enable_progress_bar = False,
        )

        t0 = time.time()
        trainer.fit(cached_student, train_dataloaders=train_loader,
                    val_dataloaders=val_loader)

        best_val_auroc = float(ckpt_cb.best_model_score) if ckpt_cb.best_model_score else None
        vauroc_str     = f"{best_val_auroc:.4f}" if best_val_auroc is not None else "na"
        print(f"  Training done in {time.time()-t0:.0f}s  (best val_unc_auroc={vauroc_str})")

        # ── Save head-only checkpoint ─────────────────────────
        ckpt_path = ckpt_dir / f"head_{ARCH}_year{YEAR}-vauroc={vauroc_str}.pt"
        torch.save({
            "uncertainty_head": cached_student.uncertainty_head.state_dict(),
            "dec_out_ch":       dec_out_ch,
            "arch":             ARCH,
            "test_year":        YEAR,
            "backbone_fold":    bb_fold,
            "backbone_pth":     backbone_pth,
            "best_val_auroc":   best_val_auroc,
        }, str(ckpt_path))
        print(f"  Saved: {ckpt_path.name}")

        results_summary.append({
            "arch":           ARCH,
            "test_year":      YEAR,
            "backbone_fold":  bb_fold,
            "backbone_ap":    bb_ap,
            "best_val_auroc": best_val_auroc,
            "checkpoint":     str(ckpt_path),
        })

        del train_dec, train_seg, train_y, teacher_unc_t
        del val_dec,   val_seg,   val_y,   val_teacher_unc_t
        del cached_student, trainer

print(f"\n{'#'*64}")
print(f"  All student training done.")
print(f"{'#'*64}")
print(json.dumps(results_summary, indent=2, default=str))
